In [5]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [6]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import numpy as np
import time
import json
from datetime import datetime, timedelta
from tqdm import tqdm

load_dotenv() 

True

In [ ]:
# 농수축산 도매시장 코드
API_KEY = os.getenv("API_KEY")
START_INDEX = 1
END_INDEX=1000


data_list = []

while START_INDEX<=17001:
    url = f"http://211.237.50.150:7080/openapi/{API_KEY}/json/Grid_20240626000000000668_1/{START_INDEX}/{END_INDEX}"
    # API 호출
    response = requests.get(url)

    # 결과 확인 및 데이터 추가
    if response.status_code == 200:
        json_data = response.json()
        data = json_data.get('Grid_20240626000000000668_1')
        data_list.extend(data.get('row'))
        print(f"API 호출 작업중 : {START_INDEX} - {END_INDEX}")
    else:
        print(f"API 호출 실패 - 코드 : {response.status_code}, 작업중 : {START_INDEX} - {END_INDEX}" )
        break

    # 다음 사이클로 이동
    START_INDEX += 1000
    if START_INDEX==17001:
        END_INDEX = 17555
    else:
        END_INDEX += 1000
    time.sleep(0.5)

df = pd.DataFrame(data_list)
df.to_csv(f'농축산_품목코드.csv', encoding='cp949')

In [ ]:
# 코드 최적화

In [13]:
# 경락데이터 가져오기
# https://data.mafra.go.kr/opendata/data/indexOpenDataDetail.do?data_id=20240625000000002460

API_KEY = os.getenv("API_KEY")
API_URL='Grid_20240625000000000654_1'
df_market = pd.read_csv('도매시장_코드.csv', encoding='cp949')

ITEM_CODES = {
    "양파": "1201",
    "배추": "1001",
    "상추": "1005",
    "사과": "0601",
    "쌀": "0103"
}

max_retries = 3
FAIL_LOG = []

# 아이템 코드 별 반복 시작
for item, icode in tqdm(ITEM_CODES.items(), desc="전체 품목 진행"):  # 품목 진행률
    LARGE = icode[:2]
    MID = icode[2:]
    data_list = []
    
    
    #날짜 설정 (수기 설정)
    start_date = '20150101'
    end_date = '20150115'
    date_str= start_date
    
    start = datetime.strptime(start_date, '%Y%m%d')
    end = datetime.strptime(end_date, '%Y%m%d')
    total_days = (end-start).days
    cnt=1
    progress=0
    print(f'{item} 작업 시작 - 진척 현황 => ')
    print('[ ', end='')
    
    #날짜에 대한 반복문 실시 / 기본 인덱스 1천개씩 
    while date_str <= end_date:
        START_INDEX = 1
        END_INDEX=1000
        

        # 마켓 별 반복문
        for mcode, market in df_market.values:
            retry_count = 0
            market_success = False  # 마켓별 성공 플래그
            while retry_count < max_retries :
                # API 호출
                url = f"http://211.237.50.150:7080/openapi/{API_KEY}/json/{API_URL}/{START_INDEX}/{END_INDEX}?SALEDATE={date_str}&WHSALCD={mcode}&LARGE={LARGE}&MID={MID}"
                

                try:
                    response = requests.get(url, timeout=5)

                    # 결과 확인 및 데이터 추가
                    if response.status_code == 200:
                        json_data = response.json()
                        data = json_data.get(API_URL)
                        rows = data.get('row', [])
                        total = data.get('totalCnt', 0)
                        print(rows)
                        if rows:
                            data_list.extend(rows)
                            market_success = True

                        # 다음 페이지로 이동
                        if total > END_INDEX:
                            START_INDEX += 1000
                            END_INDEX += 1000
                        else:
                            time.sleep(0.1)
                            #print('=', end='')  # 쌓이는지 체크
                            break

                    else:
                        print(f"API 호출 실패 - 코드 : {response.status_code}, 작업중 : {START_INDEX} - {END_INDEX}" )
                        retry_count += 1
                        break

                except requests.exceptions.ConnectTimeout:
                    print(f"[{item}] 타임아웃 발생 ({retry_count+1}/{max_retries}) {url}")
                    time.sleep(2)
                except requests.exceptions.RequestException as e:
                    print(f"[{item}] 기타 요청 예외 발생: {e} ({retry_count+1}/{max_retries})")
                    time.sleep(2)
            
                if not market_success:
                    # 실패 로그 기록
                    FAIL_LOG.append({
                        "item": item,
                        "mcode": mcode,
                        "market": market,
                        "date": date_str,
                        "url": url
                    })
                    # 현재까지 저장
                    if data_list:
                        df = pd.DataFrame(data_list)
                        df.to_csv(f'data/도매시장_실시간_경락_정보_{item}_{start_date}-{date_str}_Fail.csv', encoding='cp949', index=False)
                        print(f"[{item}] {date_str} 실패로 조기 저장됨")
                        break

                time.sleep(0.5)
        
        
        # 하루씩 날짜 더함
        type_date = datetime.strptime(date_str, '%Y%m%d')
        current_date = type_date + timedelta(days=1)
        date_str = current_date.strftime('%Y%m%d')
        cnt += 1
        time.sleep(1)
        print('=', end='')
#         if cnt%(total_days//100)==0:
#             progress += 1
#             print(f'{progress}%', end='')
        
    print(']')    
    # 데이터프레임 저장    
    if data_list:
        df = pd.DataFrame(data_list)
        df.to_csv(f'data/도매시장_실시간_경락_정보_{item}_{start_date}-{end_date}.csv', encoding='cp949')
        print(f'<--{item}_{start_date}-{end_date}_{df.shape[0]}행 작업 완료-->')
        print()
        time.sleep(3)
    else:
        print(f'{item}_{start_date}-{end_date}_데이터 없음')

# 실패 로그 저장
if FAIL_LOG:
    df_fail = pd.DataFrame(FAIL_LOG)
    df_fail.to_csv('data/fail_log.csv', index=False, encoding='cp949')
    print(f"❗ 실패 요청 {len(FAIL_LOG)}건 기록됨: data/fail_log.csv")    


전체 품목 진행:   0%|                                                                            | 0/5 [00:00<?, ?it/s]

양파 작업 시작 - 진척 현황 => 
[ []
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]

전체 품목 진행:  20%|█████████████▍                                                     | 1/5 [01:59<07:57, 119.31s/it]

=]
양파_20150101-20150115_데이터 없음
배추 작업 시작 - 진척 현황 => 
[ []
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
=[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]


전체 품목 진행:  20%|█████████████▍                                                     | 1/5 [02:15<09:00, 135.21s/it]

=

KeyboardInterrupt: 

In [12]:
data_list

[]

In [35]:
url

'http://211.237.50.150:7080/openapi/98dc49a38df4b1c4ef7d89925ba3f4417c8b0240247c08d95c02fa3acde0a5e9/json/Grid_20240625000000000654_1/1/1000?SALEDATE=20150106&WHSALCD=310101&LARGE=12&MID=01'